# Saving and loading models

_PyTorch_

**Save the state_dict (just the learned numbers), reload it into a fresh model, and call eval() — the safe, portable way.**

A trained model is two things: code (the class that defines the layers) and numbers (the learned weights). Only the numbers are precious — the code already lives in your repository.

       So PyTorch's recommended save is just the numbers: model.state_dict(), a plain dictionary mapping each parameter name to its tensor. To restore, you re-run your code to build a fresh, randomly-initialized model, then pour the saved numbers back in with load_state_dict.

---

This notebook is a practice scaffold for the **Saving and loading models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ---- a tiny model + tiny synthetic data ----
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.drop = nn.Dropout(0.5)      # behaves differently in train vs eval
        self.fc2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.fc2(self.drop(torch.relu(self.fc1(x))))

x = torch.randn(16, 4)
y = torch.randn(16, 1)

model = Net()
opt   = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.MSELoss()

# ---- train a few steps ----
model.train()
for epoch in range(20):
    opt.zero_grad()
    loss = lossf(model(x), y)
    loss.backward()
    opt.step()
print("final train loss:", round(loss.item(), 4))

# ---- inspect the state_dict: just named tensors ----
sd = model.state_dict()
print("state_dict keys:", list(sd.keys()))

# ================================================================
# 1) RECOMMENDED: save the state_dict, reload into a fresh model
# ================================================================
torch.save(model.state_dict(), "m.pt")

model.eval()                                   # original in eval mode
with torch.no_grad():
    out_original = model(x)

reloaded = Net()                               # fresh random weights
reloaded.load_state_dict(torch.load("m.pt", weights_only=True))
reloaded.eval()                                # <-- the famous must-do
with torch.no_grad():
    out_reloaded = reloaded(x)

print("outputs match:", torch.allclose(out_original, out_reloaded))   # True
# (without eval(), dropout would randomly zero units -> outputs would NOT match)

# ================================================================
# 2) CHECKPOINT: save model + optimizer + epoch + loss, then resume
# ================================================================
torch.save({
    "epoch": epoch,
    "model": model.state_dict(),
    "optim": opt.state_dict(),       # Adam's momentum/variance -> clean resume
    "loss":  loss.item(),
}, "ckpt.pt")

ckpt = torch.load("ckpt.pt", weights_only=True)
model2 = Net()
opt2   = torch.optim.Adam(model2.parameters(), lr=0.05)
model2.load_state_dict(ckpt["model"])
opt2.load_state_dict(ckpt["optim"])
start_epoch = ckpt["epoch"] + 1
print(f"resuming from epoch {start_epoch}, last loss {ckpt['loss']:.4f}")

model2.train()                                  # back to training mode to continue
for epoch in range(start_epoch, start_epoch + 5):
    opt2.zero_grad()
    loss = lossf(model2(x), y)
    loss.backward()
    opt2.step()
print("loss after resume:", round(loss.item(), 4))

# ================================================================
# 3) map_location: load a (possibly GPU-trained) file onto the CPU
# ================================================================
cpu_model = Net()
cpu_model.load_state_dict(
    torch.load("m.pt", map_location="cpu", weights_only=True)
)
cpu_model.eval()
print("loaded onto:", next(cpu_model.parameters()).device)   # cpu

## Visualize the data & results

_How big is the saved file when you store just the state_dict vs. how the bytes break down by tensor — and does the reloaded model really produce identical outputs?_

In [ ]:
import numpy as np

# ---- 1) bytes by tensor: param-count -> bytes (float32 = 4 bytes) ----
shapes = {
    "fc1.weight": (8, 4),
    "fc1.bias":   (8,),
    "fc2.weight": (1, 8),
    "fc2.bias":   (1,),
}
BYTES_PER_F32 = 4
sizes = {k: int(np.prod(s)) * BYTES_PER_F32 for k, s in shapes.items()}
total_params = sum(int(np.prod(s)) for s in shapes.values())
print("bytes by tensor:", sizes)               # {'fc1.weight':128,'fc1.bias':32,'fc2.weight':32,'fc2.bias':4}
print("total params:", total_params,           # 97
      "-> raw bytes:", total_params * BYTES_PER_F32)   # 388

# ---- 2) reload check: identical with eval(), differs without ----
rng = np.random.default_rng(0)
# stand in for the model's pre-dropout activations (8 hidden units, 16 examples)
hidden = np.abs(rng.standard_normal((16, 8)))      # relu output >= 0
w2 = rng.standard_normal((8, 1))                   # fc2.weight

# eval(): dropout OFF -> reloaded weights give the SAME output as the original
out_eval = hidden @ w2
diff_with_eval = float(np.max(np.abs(out_eval - out_eval)))   # 0.0 (same weights, same math)

# train mode: dropout ON -> units randomly zeroed and scaled by 1/(1-p)
p = 0.5
mask = (rng.random((16, 8)) > p) / (1 - p)
out_train = (hidden * mask) @ w2
diff_no_eval = float(np.max(np.abs(out_eval - out_train)))    # large, non-zero
print("max diff WITH eval():", round(diff_with_eval, 4))      # 0.0
print("max diff NO eval()  :", round(diff_no_eval, 4))        # ~0.83

# ---- charts ----
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(list(sizes.keys()), list(sizes.values()),
          color=["#4ea1ff", "#7ee787", "#c89bff", "#ffa657"])
ax[0].set_ylabel("bytes"); ax[0].set_title("state_dict size by tensor (float32)")
ax[0].tick_params(axis="x", rotation=20)
ax[1].bar(["with eval()", "no eval()"], [diff_with_eval, diff_no_eval],
          color=["#7ee787", "#ff7b72"])
ax[1].set_ylabel("max |orig - reloaded|")
ax[1].set_title("reloaded outputs match only after eval()")
plt.tight_layout(); plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Define a tiny Net with fc1 = nn.Linear(4, 8) and fc2 = nn.Linear(8, 1). Build one, then print list(model.state_dict().keys()). Predict the four key names before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call model.state_dict().keys(). — _The state_dict is a dictionary keyed by dotted parameter paths._
- Read the names. — _Each submodule's params are namespaced: fc1.weight, fc1.bias, etc._

**Answer:** import torch
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
model = Net()
print(list(model.state_dict().keys()))
# ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']

</details>

**Problem 2.** Type this in Colab. Save just the numbers, not the object. Save model.state_dict() to "m.pt" with torch.save. Then build a fresh Net() (random weights), load the saved dict into it with weights_only=True, and confirm load_state_dict reports no missing/unexpected keys.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- torch.save(model.state_dict(), "m.pt") — the state_dict, not model itself. — _Saving the whole object pickles the class by reference and breaks on refactors._
- fresh.load_state_dict(torch.load("m.pt", weights_only=True)). — _It returns <All keys matched successfully> when the architectures align._

**Answer:** torch.save(model.state_dict(), "m.pt")
fresh = Net()                                    # random weights
result = fresh.load_state_dict(torch.load("m.pt", weights_only=True))
print(result)   # <All keys matched successfully>

</details>

**Problem 3.** Type this in Colab. The famous eval()-after-load rule. Add a nn.Dropout(0.5) between the layers of Net. Save and reload the weights. Feed the same input x = torch.randn(4, 4) through the original (in eval) and the reloaded model — show outputs match only after calling reloaded.eval().

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Put both models in eval() and compare with torch.allclose. — _eval() turns dropout off so the math is deterministic._
- Then put the reloaded model in train() and compare again. — _In training mode dropout randomly zeroes units, so outputs diverge._

**Answer:** torch.manual_seed(0)
class NetD(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8); self.drop = nn.Dropout(0.5); self.fc2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.fc2(self.drop(torch.relu(self.fc1(x))))
m = NetD(); torch.save(m.state_dict(), "d.pt")
x = torch.randn(4, 4)
m.eval()
with torch.no_grad(): orig = m(x)
r = NetD(); r.load_state_dict(torch.load("d.pt", weights_only=True))
r.eval()
with torch.no_grad(): print(torch.allclose(orig, r(x)))   # True
r.train()
with torch.no_grad(): print(torch.allclose(orig, r(x)))   # False (dropout on)

</details>

**Problem 4.** Type this in Colab. Device mismatch on load. Simulate a GPU-trained file by loading with map_location="cpu". Load "m.pt" remapping every tensor to the CPU, load it into a fresh Net(), and print the device of the first parameter.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass map_location="cpu" to torch.load. — _It redirects every saved tensor to the CPU as it deserializes, before any absent GPU is touched._
- Check next(model.parameters()).device. — _Confirms the loaded weights landed on the CPU._

**Answer:** cpu_model = Net()
cpu_model.load_state_dict(
    torch.load("m.pt", map_location="cpu", weights_only=True)
)
cpu_model.eval()
print(next(cpu_model.parameters()).device)   # cpu

</details>

**Problem 5.** Type this in Colab. Save a full checkpoint for resuming. Train Net a few steps with Adam, then save a dict containing the model state_dict, optimizer state_dict, the epoch, and the last loss into "ckpt.pt". Print the dict's keys.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the dict {"epoch": e, "model": model.state_dict(), "optim": opt.state_dict(), "loss": loss.item()}. — _Adam keeps per-parameter momentum/variance; the optimizer state must travel too._
- torch.save(ckpt, "ckpt.pt") and print ckpt.keys(). — _A checkpoint is just a bigger dictionary saved with the same call._

**Answer:** torch.manual_seed(0)
model = Net(); opt = torch.optim.Adam(model.parameters(), lr=0.05)
lossf = nn.MSELoss(); x = torch.randn(16, 4); y = torch.randn(16, 1)
for e in range(10):
    opt.zero_grad(); loss = lossf(model(x), y); loss.backward(); opt.step()
ckpt = {"epoch": e, "model": model.state_dict(), "optim": opt.state_dict(), "loss": loss.item()}
torch.save(ckpt, "ckpt.pt")
print(list(ckpt.keys()))   # ['epoch', 'model', 'optim', 'loss']

</details>

**Problem 6.** Type this in Colab. Resume from the checkpoint. Load "ckpt.pt", push the model and optimizer states back into fresh objects, set start_epoch = ckpt["epoch"] + 1, call model.train(), and print the resume epoch. Predict it before running (the save loop ended at epoch 9).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Restore both states: model2.load_state_dict(ckpt["model"]) and opt2.load_state_dict(ckpt["optim"]). — _Restoring the optimizer keeps Adam's running averages warm so resumed steps do not lurch._
- Set start_epoch = ckpt["epoch"] + 1 and call model2.train(). — _Resume at the next epoch, in training mode._

**Answer:** ckpt = torch.load("ckpt.pt", weights_only=True)
model2 = Net(); opt2 = torch.optim.Adam(model2.parameters(), lr=0.05)
model2.load_state_dict(ckpt["model"])
opt2.load_state_dict(ckpt["optim"])
start_epoch = ckpt["epoch"] + 1
model2.train()
print(start_epoch)   # 10  (saved at epoch 9)

</details>

**Problem 7.** Type this in Colab. Partial load with strict=False (transfer-learning style). Take the saved Net weights but load them into a model whose fc2 has a different output size, using strict=False. Print the returned missing and unexpected keys.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build a variant with fc2 = nn.Linear(8, 3) and load with strict=False. — _Strict loading would raise on the shape mismatch; strict=False copies only the keys that match._
- Inspect the returned missing_keys / unexpected_keys. — _It reports which tensors did not get copied so you can spot mismatches._

**Answer:** class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8); self.fc2 = nn.Linear(8, 3)   # different head
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
out = Net3().load_state_dict(torch.load("m.pt", weights_only=True), strict=False)
print(out.missing_keys)      # [] (fc1/fc2 names exist) -- but fc2 shapes differ
# In practice the size-mismatched fc2.* are skipped; matching fc1.* are copied.
# missing/unexpected lists report any names that did not line up.

</details>

**Problem 8.** Type this in Colab. Three-line inference reload. Your teammate sends m.pt (a state_dict). Write the canonical three-line load-for-inference: rebuild the class, load_state_dict with weights_only=True, and eval(). Then run a no-grad forward on torch.randn(2, 4).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- model = Net(); model.load_state_dict(torch.load("m.pt", weights_only=True)); model.eval(). — _Rebuild code, pour in numbers, switch off dropout/batch-norm training behavior._
- Wrap inference in torch.no_grad(). — _No autograd graph is needed at inference; it saves memory._

**Answer:** model = Net()
model.load_state_dict(torch.load("m.pt", weights_only=True))
model.eval()
with torch.no_grad():
    print(model(torch.randn(2, 4)).shape)   # torch.Size([2, 1])

</details>